# E008 trên V14 — Colab Pro

**Upload bundle trước khi kết nối GPU.** Đưa `e008-colab-bundle.zip` vào **My Drive/E008/**, giữ private. Notebook đã pin SHA; không sửa code/config/bundle giữa campaign.

1. Chọn **Runtime → Change runtime type → GPU** mạnh nhất đang được cấp (A100/H100 nếu có; L4 cũng được nếu smoke đạt). High RAM tùy khả dụng. Không mua thêm credits. Pro không bảo đảm một loại GPU hoặc số giờ cố định: [Colab FAQ](https://research.google.com/colaboratory/faq.html).
2. **Run all**, cấp quyền mount Drive đúng tài khoản. Bộ đếm **4 giờ** bắt đầu khi chạy cell, gồm setup và thời gian chờ; kết nối GPU rồi chạy ngay.
3. Notebook kiểm bundle/data/source/weights, cài environment và chạy **G0/A1 smoke full + resume 5 epoch**. Chỉ khi parity/reload/timing PASS mới mở research **G0/G1/A1**, rồi champion + G0 qua hai seed tiếp theo; tối đa **7 research run** (5 nếu G0 thắng).
4. Mỗi research run tối đa **24 epoch**, patience 6, warmup 3; FP32, 336 px, batch 16. Run + reload tối đa 25 phút; thiếu budget, OOM hoặc sai parity thì dừng, không tự đổi variables.

Kết quả: **My Drive/E008/E008-V14-COLAB-20261024/**. `runs/campaign.json` cho biết FINISHED/STOPPED; checkpoints, config, predictions, metrics và log giữ ở đây. Epoch recovery đồng bộ sang `recovery/`. Runtime tự ngắt khi xong hoặc lỗi.

Sau khi chạy, tải **My Drive/E008/E008-V14-results-metadata.zip** về để audit. File này chỉ chứa metadata/logs; model vẫn ở campaign folder. Nếu runtime bị cắt trước bước đóng gói, giữ folder, `campaign.json`, log và recovery; không bấm chạy lại toàn notebook vào identity đã tồn tại.

V14 có 1.199 train / 156 val / 0 test. Val là development và chỉ có **29 looking-positive**, dưới support gate 30: chưa thể công bố đạt đầy đủ quality gate dù metrics cao. Không test/promotion. G0 mới trên cùng V14 là baseline, không so trực tiếp với E007/V13.


In [ ]:
import hashlib, json, os, subprocess, sys, threading, time, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive, runtime

EXPECTED_SHA256 = '8d1b7f25d5fcc11dfe963ab8deea5948e5d5b0d5a3f6a3a086d606a8e116aab7'
started_at = time.time()
watchdog = threading.Timer(4 * 3600, runtime.unassign)
watchdog.daemon = True
watchdog.start()
storage = None
try:
    if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
        raise RuntimeError('This approved environment requires Python 3.11 or 3.12')
    subprocess.run(['nvidia-smi'], check=True)
    drive.mount('/content/drive')
    storage = Path('/content/drive/MyDrive/E008')
    archive_path = storage / 'e008-colab-bundle.zip'
    digest = hashlib.sha256()
    with archive_path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    if digest.hexdigest() != EXPECTED_SHA256:
        raise RuntimeError('Bundle checksum mismatch; stop without executing its code')
    root = Path('/content/e008')
    if root.exists() or (storage / 'E008-V14-COLAB-20261024').exists():
        raise RuntimeError('Existing campaign: preserve it and use explicit recovery')
    root.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        manifest = json.loads(archive.read('bundle-manifest.json'))
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(manifest['files']) | {'bundle-manifest.json'}:
            raise RuntimeError('Unexpected bundle inventory')
        for name, expected in manifest['files'].items():
            path = PurePosixPath(name)
            if path.is_absolute() or '..' in path.parts or '\\' in name or ':' in name:
                raise RuntimeError('Unsafe bundle path')
            payload = archive.read(name)
            if hashlib.sha256(payload).hexdigest() != expected:
                raise RuntimeError('Bundle member checksum mismatch: ' + name)
            destination = root / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(payload)
        (root / 'bundle-manifest.json').write_text(json.dumps(manifest, indent=2))
    setup_log = storage / ('setup-' + str(int(started_at)) + '.log')
    venv = Path('/content/e008-venv')
    python = venv / 'bin/python'
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    os.environ['XFORMERS_DISABLED'] = '1'
    with setup_log.open('w') as log:
        commands = [
            [sys.executable, '-m', 'venv', str(venv)],
            [str(python), '-m', 'pip', 'install', '-r', 'requirements/classifier-colab.txt'],
            [str(python), '-m', 'pip', 'install', '--no-deps', '-e', '.'],
        ]
        for command in commands:
            subprocess.run(command, cwd=root, stdout=log, stderr=subprocess.STDOUT, check=True)
    freeze = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], cwd=root, text=True)
    (storage / ('environment-' + str(int(started_at)) + '.txt')).write_text(freeze)
    print('Setup complete. Logs and epoch recovery are saved to private Drive.', flush=True)
    subprocess.run([str(python), '-m', 'ai_exam_monitoring.training.representation_campaign',
                    '--workspace', str(root), '--storage', str(storage),
                    '--design', str(root / 'configs/experiments/E008-V14-LOOKING-20261024/design.json'),
                    '--gpu-started-at', str(started_at)], cwd=root, check=True)
    print('Campaign finished. Review campaign.json and metrics in Drive; no model promotion implied.')
except BaseException as error:
    if storage is not None and storage.is_dir():
        (storage / ('launcher-error-' + str(int(started_at)) + '.txt')).write_text(repr(error))
    raise
finally:
    try:
        if storage is not None and storage.is_dir():
            campaign = storage / 'E008-V14-COLAB-20261024'
            if campaign.is_dir():
                with zipfile.ZipFile(storage / 'E008-V14-results-metadata.zip', 'w',
                                     compression=zipfile.ZIP_DEFLATED) as results:
                    for path in sorted(campaign.rglob('*')):
                        if path.is_file() and path.suffix in {'.json', '.yaml', '.log', '.txt'}:
                            results.write(path, path.relative_to(campaign).as_posix())
                print('Metadata: My Drive/E008/E008-V14-results-metadata.zip', flush=True)
    finally:
        watchdog.cancel()
        runtime.unassign()
